
# Generative AI and Modern Applications

## Text Generation: It's Just Statistics

At its core, a Generative LLM is a **Next-Token Prediction Engine**. It doesn't "know" facts; it knows probabilities. Given the sequence "The sky is", it calculates the probability distribution for the next word:

-   "blue": 80%
-   "gray": 15%
-   "falling": 1%

It samples from this distribution to generate text.

## Retrieval-Augmented Generation (RAG)

LLMs have two major flaws:

1.  **Hallucination**: They make things up.
2.  **Cut-off Date**: They don't know current events or private company data.

**RAG** solves this by connecting the LLM to an external library (like a Vector Database).

-   **Step 1 (Retrieve)**: User asks a question. System searches the database for relevant documents.
-   **Step 2 (Augment)**: System pastes those documents into the prompt context.
-   **Step 3 (Generate)**: LLM answers the question using the provided context.

### What are Embeddings? (The Secret Sauce)

Before diving into RAG, we need to understand \*Embeddings\*—vectors that capture meaning.

-   An **Embedding** is a list of numbers (e.g., 768 floats) representing a word, sentence, or document.
-   **Key Property**: Similar meanings → Similar vectors (measured by cosine similarity).
-   "King" and "Queen" are close in embedding space; "King" and "Banana" are far apart.

Modern RAG uses **Sentence Embeddings** (from models like BERT) instead of keyword matching. This allows semantic search: "What's the price?" matches "Product A costs $50" even without shared words.

## Practical Demonstration: Building a "Tiny RAG"

We will simulate a RAG system that answers questions about a fictional company using a private knowledge base.

### The Knowledge Base

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Our "Private" Database (LLM doesn't know this)
documents = [
    "Product A costs $50 and includes a 1-year warranty.",
    "Product B is a subscription service costing $10/month.",
    "Support is available 24/7 via email at help@example.com.",
    "The CEO of the company is Jane Doe."
]

# Indexing (Simple TF-IDF)
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

print(f"Indexed {len(documents)} documents.")

### The Retrieval System

When a user asks a question, we find the most similar document.

In [ ]:
def retrieve_info(query):
    # Vectorize query
    query_vec = vectorizer.transform([query])

    # Calculate similarity
    similarities = cosine_similarity(query_vec, doc_vectors).flatten()

    # Find best match
    best_idx = np.argmax(similarities)
    return documents[best_idx]

user_query = "How much does Product A cost?"
retrieved_doc = retrieve_info(user_query)

print(f"Query: {user_query}")
print(f"Retrieved Context: {retrieved_doc}")

### The Limitation of Keyword Matching

TF-IDF matches **words**, not **meaning**. Watch it fail on a paraphrased query:

In [ ]:
# Same question, different words
queries = [
    "How much does Product A cost?",    # Direct match (works)
    "What's the price of Product A?",   # Paraphrased (may fail)
    "Tell me about the warranty",       # Different focus
]

print("TF-IDF Retrieval Results:")
print("-" * 50)
for q in queries:
    result = retrieve_info(q)
    print(f"Q: {q}")
    print(f"→ {result[:50]}...")
    print()

**Observation**: TF-IDF struggles with synonyms ("cost" vs "price"). Real RAG systems use **semantic embeddings** (sentence-transformers) that understand meaning, not just keywords.

### The Generation (Augmented Prompt)

Now we feed this to the LLM.

In [ ]:
from transformers import pipeline

# Load a small text generator
generator = pipeline('text-generation', model='gpt2')

# Construct the Augmented Prompt
prompt = f"Context: {retrieved_doc}\nQuestion: {user_query}\nAnswer:"

output = generator(prompt, max_new_tokens=10, pad_token_id=50256)
print(f"--- Final LLM Output ---\n{output[0]['generated_text']}")

**Observation**: The LLM answers correctly because we **gave** it the answer in the context.

## Introduction to Diffusion Models (Image Gen)

Models like **Stable Diffusion** and **DALL-E** work differently from LLMs. They rely on **Diffusion**.

### How it works

1.  **Forward Process (Training)**: Take a clear image and slowly add "static" (Gaussian Noise) until it is pure random noise. The model watches this and learns the mathematical pattern of how the image was destroyed.
2.  **Reverse Process (Generation)**: Start with pure random noise. Ask the model: **"If this was a picture of a cat, how would you remove the noise?"** Repeat this 50 times until a clear image emerges.

### How Text Guides Generation (Conditioning)

But how does "a cat wearing a hat" become an image? The secret is **Conditioning**:

1.  **CLIP Encoder**: A separate model (trained on image-text pairs) converts your text prompt into an embedding vector.
2.  **Guided Denoising**: At each step, the diffusion model asks: "What noise should I remove to make this image **more similar** to the text embedding?"
3.  **Iterative Refinement**: After 20-50 steps, the noise becomes a coherent image matching your description.

This is why prompt engineering matters for image generation too—different phrasings produce different CLIP embeddings, leading to different images.

## Practical Demonstration: The Forward Process

We can visualize how the model learns by destroying an image ourselves.

In [ ]:
import matplotlib.pyplot as plt
from skimage import data
import numpy as np

# Load sample image (Astronaut)
img = data.astronaut()

# Normalize to 0-1
img = img / 255.0

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(img)
axes[0].set_title("Step 0 (Clean)")
axes[0].axis('off')

# Simulate Forward Diffusion (Adding Noise)
current_img = img.copy()
noise_level = 0.3 # How much noise to add per step

for i in range(1, 5):
    # Add Gaussian Noise
    noise = np.random.normal(0, noise_level, img.shape)
    current_img = current_img + noise

    # Clip to stay valid color range
    current_img = np.clip(current_img, 0, 1)

    axes[i].imshow(current_img)
    axes[i].set_title(f"Step {i*25} (Noisier)")
    axes[i].axis('off')

plt.suptitle("Forward Diffusion: What the Model Learns to Undo")
plt.show()

## Ethical Considerations and Bias

With great power comes great responsibility.

### Bias

GenAI models are trained on the internet. The internet contains stereotypes, racism, and sexism.

-   **Example**: If you ask an image generator for a "Doctor," it might generate only men. If you ask for a "Nurse," it might generate only women.
-   **Mitigation**: Curating training data and using "RLHF" (Human Feedback) to penalize biased outputs.

### Hallucination

LLMs are confident liars.

-   They optimize for **plausibility**, not **truth**.
-   **Danger**: Using LLMs for medical or legal advice without verification (RAG helps reduce this).

### Copyright

-   Image models are trained on billions of copyrighted images (artists' work).
-   Code models are trained on GitHub repositories.
-   The legal framework for this is still being debated globally.

## Exercises

### RAG Query Robustness

Test how well TF-IDF handles different query phrasings.

1.  Add more documents to the knowledge base.
2.  Try queries with synonyms, typos, or completely different phrasing.
3.  Which queries fail? Why?

### Diffusion Noise Schedule

The rate at which noise is added affects image quality.

1.  Modify the `noise_level` parameter in the diffusion demo (try 0.1, 0.5, 1.0).
2.  How does the "destruction speed" change?
3.  Real diffusion models use carefully tuned **noise schedules** (linear, cosine, etc.).

### Emerging Paradigm: AI Agents

Modern GenAI goes beyond chat. **AI Agents** combine LLMs with tools:

-   **Code Execution**: The LLM writes Python and runs it.
-   **Web Search**: The LLM searches the internet for current information.
-   **Tool Use**: The LLM calls APIs (calculators, databases, etc.).

This is the frontier of GenAI—systems that **act**, not just **respond**.

## Summary

1.  **Next-Token Prediction**: The simple math behind the complex magic of LLMs.
2.  **RAG**: The bridge between a generic LLM and your private data.
3.  **Diffusion**: Creating images by learning to reverse-engineer noise.
4.  **Ethics**: AI reflects the data it was fed. It requires human oversight.